# OWL 2 RL materialization with the Eyeling ruleset

This notebook runs the complete [OWL 2 RL/RDF ruleset maintained for Eyeling](https://github.com/pietercolpaert/rdfjs-inference-engine/blob/main/rules/owl2rl/owl2rl-eyeling.n3), rather than a hand-written subset. The profile is an N3 materialization ruleset: it covers OWL 2 RL rules that are expressible in Eyeling, including datatype handling, RDF-list helpers, property chains, and explicit inconsistency resources.

In [1]:
from urllib.request import Request, urlopen

from pyling import reason_stream

OWL2RL_URL = "https://raw.githubusercontent.com/pietercolpaert/rdfjs-inference-engine/refs/heads/main/rules/owl2rl/owl2rl-eyeling.n3"


def load_text(url):
    request = Request(url, headers={"User-Agent": "pyling-notebook"})
    with urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8")


owl2rl_rules = load_text(OWL2RL_URL)
print(f"Loaded {len(owl2rl_rules.splitlines()):,} lines of OWL 2 RL N3 rules")

Loaded 1,174 lines of OWL 2 RL N3 rules


The input exercises several parts of the real profile at once: transitive subclass inference, property domain and range, inverse properties, symmetric properties, and a transitive object property.

In [2]:
data = """
@prefix : <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:authored rdfs:domain :Researcher ;
          rdfs:range :Paper ;
          owl:inverseOf :hasAuthor .
:collaboratesWith a owl:SymmetricProperty .
:ancestorOf a owl:TransitiveProperty .
:Researcher rdfs:subClassOf :Person .
:Person rdfs:subClassOf :Agent .

:alice :authored :paper42 ;
       :collaboratesWith :bob ;
       :ancestorOf :bob .
:bob :ancestorOf :carol .
"""

result = reason_stream(
    {"sources": [owl2rl_rules, data]},
    include_input_facts_in_closure=True,
)
print(f"Derived {len(result.derived)} triples")

Derived 58 triples


In [3]:
expected = [
    ":alice a :Researcher .",
    ":alice a :Person .",
    ":alice a :Agent .",
    ":paper42 a :Paper .",
    ":paper42 :hasAuthor :alice .",
    ":bob :collaboratesWith :alice .",
    ":alice :ancestorOf :carol .",
]

for triple in expected:
    assert triple in result.closure_n3, triple
    print(triple)

:alice a :Researcher .
:alice a :Person .
:alice a :Agent .
:paper42 a :Paper .
:paper42 :hasAuthor :alice .
:bob :collaboratesWith :alice .
:alice :ancestorOf :carol .


These consequences come from the downloaded profile itself. This is OWL 2 RL forward materialization, not an OWL 2 DL tableau reasoner; the ruleset also represents detected inconsistencies as explicit resources so applications can inspect them.